# MLIP examples: Hessian Model Training

This notebook builds on top of the [regular training tutorial](https://github.com/instadeepai/mlip/blob/main/tutorials/model_training_tutorial.ipynb) to provide an example on how to use the library to train Hessian-aware MLIP models. Second-order derivatives of the energy with respect to positions are essential predictions in many
downstream tasks, either explicitly (e.g., vibrational frequencies or zero point energy) or implicitly via running Hessian estimates. As a result we incoporate Hessian training and calculation options for models built in *mlip*. 

As an efficient alternative to direclty learning the full Hessian, we only differentiate randomly selected force components with respect to atomic coordinates, an approach used for example in <sup>[1]</sup>. Further details on the implementation can be found in our whitepaper. 

**It covers the following aspects:**

- **How to prepare a dataset** for training and validation on Hessian-labeled data.
- **How to predict Hessians** with any MLIP model (even if not trained on Hessians).
- **How to train an MLIP network** with Hessian supervision.
- **How to calculate full Hessian matrices** using a Hessian-trained model, and presenting some comparative results.

**NB**: This notebook assumes the reader is already familiar with the basic [data processing documentation](https://instadeepai.github.io/mlip/user_guide/data_processing.html) of *mlip*.

---
<sup>[1]</sup> Ishan Amin, Sanjeev Raja, and Aditi Krishnapriyan. Towards fast, specialized machine learning force fields: Distilling foundation models via energy hessians. arxiv 2025, doi: 10.48550. arXiv preprint arXiv.2501.09009, 2025.

**Install, required imports, and logging setup**


We will start by installing the *mlip* library directly from pip, setup logging, and import required modules and helpers, exactly as shown in the [regular training tutorial](https://github.com/instadeepai/mlip/blob/main/tutorials/model_training_tutorial.ipynb).


In [ ]:
%pip install "mlip[cuda]" huggingface_hub matplotlib

# Use this instead for installation without GPU:
# %pip install mlip huggingface_hub matplotlib

In [ ]:
# Other
import dataclasses
import functools
import logging

import jax
import matplotlib.pyplot as plt
import numpy as np

# For dataset loading & processing
from mlip.data import ExtxyzReader, GraphDatasetBuilder, Hdf5Reader
from mlip.data.configs import GraphDatasetBuilderConfig
from mlip.data.graph_dataset_builder import BuilderMode
from mlip.data.helpers.combined_graph_dataset import CombinedGraphDataset
from mlip.data.helpers.hessian_utils import pad_systems_hessians, process_graph_hessian

# For model instantiation
from mlip.models import ForceField, Visnet

# For loss function
from mlip.models.loss import MSELoss

# For optimizer & training
from mlip.training import TrainingLoop, TrainingLoopConfig, get_default_mlip_optimizer
from mlip.typing.properties import Properties

In [ ]:
logging.basicConfig(
    level=logging.INFO, force=True, format="%(levelname)s - %(message)s"
)
logger = logging.getLogger("mlip")

In [ ]:
print(jax.devices())

## 1. Preparing a Hessian-labeled dataset

Generally, Hessian reference data generated with quantum chemistry methods come in small sizes, due to the computational cost of performing Density Functional Theory (DFT) calculations to obtain the second-order derivatives of the energy Hessian with respect to atomic positions, especially at higher levels of theory (LoT), and for larger molecules.

In order to efficiently mix and train on datasets with (usually the smaller portion of the whole data) and without Hessian labels, we introduce the [`CombinedGraphDataset`]('https://instadeepai.github.io/mlip/api_reference/data/combined_graph_dataset.html') class which can be used as shown below:

### 1.1 Download the Hessian-labeled and non-Hessian-labeled data files to the local workspace

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="InstaDeepAI/MLIP-tutorials",
    allow_patterns=["hessian_training/*", "training/*"],
    local_dir="",
)

### 1.2 Create separate `readers` dictionaries for each dataset type (w/ & w/o Hessians)

Then we create two separate reader dictionaries, one for each dataset type. Both dictionaries have to respect the following format:
```
Dict[str, List[ChemicalSystemsReader]]
i.e. {split_name: [ChemicalSystemsReader, ...]}
```
Our `Hdf5` containing the Hessian labels stores the property under the `hessian_matrix` key, whereas by default, the [`Hdf5Reader`](https://instadeepai.github.io/mlip/api_reference/data/chemical_systems_readers/hdf5_reader.html) will look for the `hessian` key. Therefore, we need to pass-in our custom `property_name_mapping` upon creation of the reader objects. One can also control the number of structures to load from the input file, by setting `num_to_load` in the reader's constructor.

Note that currently, Hessian labels are only supported to be stored in `Hdf5` files, but one can freely extend the library with a custom reader to digest other file formats. 

In [ ]:
property_name_mapping = {"hessian": "hessian_matrix"}

hessian_subset_readers = {
    "train": [
        Hdf5Reader(
            "hessian_training/rmd17_aspirin_hessians_train.hdf5",
            property_name_mapping=property_name_mapping,
        )
    ],
    "valid": [
        Hdf5Reader(
            "hessian_training/rmd17_aspirin_hessians_val.hdf5",
            property_name_mapping=property_name_mapping,
        )
    ],
}

non_hessian_subset_readers = {
    "train": [ExtxyzReader("training/rmd17_aspirin_train.xyz")],
    "valid": [ExtxyzReader("training/rmd17_aspirin_val.xyz")],
}

**Important**: the Hessian data processing workflow only works with exactly two subsets, and consequently two subset reader dictionaries, one for data with Hessians and one without.

### 1.3 Create [`GraphDatasetBuilder`](https://instadeepai.github.io/mlip/api_reference/data/graph_dataset_builder.html) instances from the reader dictionaries

In [ ]:
graph_cutoff_angstrom = 5.0
non_hessian_subset_builder_config = GraphDatasetBuilderConfig(
    graph_cutoff_angstrom=graph_cutoff_angstrom,
    batch_size=16,
)
hessian_subset_builder_config = GraphDatasetBuilderConfig(
    graph_cutoff_angstrom=graph_cutoff_angstrom,
    batch_size=4,
)

In [ ]:
non_hessian_subset_builder = GraphDatasetBuilder(
    builder_config=non_hessian_subset_builder_config,
    readers=non_hessian_subset_readers,
    mode=BuilderMode.TRAINING,
    dataset_info=None,
)
non_hessian_subset_splits = non_hessian_subset_builder.get_datasets(prefetch=False)

To ensure the compatibility between both datasets we are aiming to combine, we use the [`DatasetInfo`](https://instadeepai.github.io/mlip/api_reference/data/dataset_info.html) object of the non-Hessian [`GraphDatasetBuilder`](https://instadeepai.github.io/mlip/api_reference/data/graph_dataset_builder.html) (as it would generally correspond to the larger dataset among the two) to create the Hessian-annotated dataset builder.

In [ ]:
hessian_subset_builder = GraphDatasetBuilder(
    builder_config=hessian_subset_builder_config,
    readers=hessian_subset_readers,
    mode=BuilderMode.TRAINING,
    dataset_info=non_hessian_subset_builder.dataset_info,
)

*mlip* provides the user with the option to easily pass-in arbitrary custom data processing functions. We use this functionality to apply the necessary pre/post processing to allow for a **jittable and batched subsampling-based Hessian training**:



*   **`pad_systems_hessians()`**: pads all Hessian matrices to the maximum system size `N`, transforming shapes from `(n, 3, n, 3)` to `(n, 3, N, 3)` (`n` is the number of atoms for a given system) to enable the batching of Hessians with heterogeneous shapes.

*   **`process_graph_hessian()`**: will be applied on the fly as batches are loaded during training. It samples `R` random Hessian rows from each graph in the batch. Has to be partially initialized with `num_rows` (recommended=`{2, 4, 8, 16}`)

In [ ]:
hessian_subset_splits = hessian_subset_builder.get_datasets(
    prefetch=False,
    systems_preprocessing=[pad_systems_hessians],
    graph_postprocessing=[functools.partial(process_graph_hessian, num_rows=4)],
)

### 1.4 Merge train and validation splits by wrapping splits inside [`CombinedGraphDataset`](https://instadeepai.github.io/mlip/api_reference/data/combined_graph_dataset.html) objects

Creating the [`CombinedGraphDataset`](https://instadeepai.github.io/mlip/api_reference/data/combined_graph_dataset.html) objects with `interleaving_method=regular` will apply a deterministic interleaving approach preserving an ordering compatible with a parallel training setup. Each group of `N` consecutive items must be homogeneous, i.e, all items in the group must come from the same source iterable (w/ or w/o Hessians). To enforce this, the generator interleaves items as follows: after every
`r * N` items drawn from `ds1 (large)`, `N` items are drawn from
`ds2 (small)` (`r` represents the ratio between dataset sizes, and `N` the number of allocated devices). Both sequences are therefore consumed in chunks that are
divisible by `N`, ensuring that each `N` loaded batches are items from only
one iterable.

In [ ]:
combined_splits = {
    split_name: CombinedGraphDataset.init(
        graph_datasets=[
            non_hessian_subset_splits[split_name],
            hessian_subset_splits[split_name],
        ],
        interleaving_method="regular",
    )
    for split_name in ["train", "valid"]
}

In [ ]:
logger.info(f"Dataset info: {non_hessian_subset_builder.dataset_info}")
for split_name, dataset in combined_splits.items():
    split_len = (
        len(dataset)
        if not isinstance(dataset, dict)
        else sum(map(len, dataset.values()))
    )

    logger.info("Number of batches in %s set: %s", split_name, split_len)

## 2. Predicting Full Hessians from any MLIP model

Prior to showing how one can train a model on Hessians, it is worth noting that the library is set-up to allow users to predict Hessians from any pretrained model. To demonstrate this, we start by training a small model (for full details, please refer to our [regular model training tutorial](https://github.com/instadeepai/mlip/blob/main/tutorials/model_training_tutorial.ipynb)). 

In [ ]:
mlip_network_no_hessians = Visnet(
    Visnet.Config(num_layers=1, num_channels=4, num_heads=2, num_rbf=2),
    non_hessian_subset_builder.dataset_info,
)

force_field_no_hessians = ForceField.from_mlip_network(
    mlip_network_no_hessians, required_properties=Properties(hessian=False), seed=42
)

optimizer = get_default_mlip_optimizer()
loss = MSELoss(extended_metrics=True)

training_config = TrainingLoopConfig(num_epochs=10)

training_loop_no_hessians = TrainingLoop(
    train_dataset=combined_splits["train"],
    validation_dataset=combined_splits["valid"],
    force_field=force_field_no_hessians,
    loss=loss,
    optimizer=optimizer,
    config=training_config,
)

In [ ]:
training_loop_no_hessians.run()

Once the model is trained, a few steps are required to allow for the model to provide a prediction on Hessians from a given input structure. These may appear a little complicated compared to the rest of the API, but this is to guarantee flexibility around developing novel methods for training on partial Hessian labels. Further details on this design choice can be found in the [library documentation](https://instadeepai.github.io/mlip/).

We start by selecting a reference graph from the validation set. 

In [ ]:
# Take a graph from the validation dataset
example_graph = hessian_subset_splits["valid"].graphs[0]

ref_hessian = example_graph.nodes.hessian  # we can extract the reference hessian easily

We then need to modify the example graph in order to remove the Hessian data (in a normal inference task, outside of validation, this would not be necessary)

In [ ]:
# Remove reference Hessian from graph
example_graph = example_graph.replace_nodes(hessian=None)

# Important: we need to change the global properties of the graph
# so it carries the information that a full Hessian calculation is
# required and not just a subsample of Hessian rows
example_graph = example_graph.request_full_hessian()

Final step is to modify the `Predictor` wrapping the trained MLIP model and then run the predictions:

In [ ]:
# 1. Extract the best model from the training loop
ff_no_hessian = training_loop_no_hessians.best_model

# 2. Update the predictor for the model to predict Hessians
ff_no_hessian = ff_no_hessian.replace_required_properties(Properties(hessian=True))

# 3. Run the prediction task
jitted_force_field_fun_no_hessians = jax.jit(ff_no_hessian)
prediction_no_hessians = jitted_force_field_fun_no_hessians(example_graph)

# 4. We can quickly check the Hessian shape:

print(f"Predicted Hessian matrix shape: {prediction_no_hessians.hessian.shape}")

## 3. Preparing and running a training loop for Hessian models

We prepare the Hessian-training loop exactly as we would do for a regular training, except, we need to explicitly set the `hessian` property as required, so that the [`MLIPNetwork`](https://instadeepai.github.io/mlip/api_reference/models/mlip_network.html) is wrapped inside a [`HessianPredictor`](https://instadeepai.github.io/api_reference/models/hessian_predictor.html).

In [ ]:
mlip_network = Visnet(
    Visnet.Config(num_layers=1, num_channels=4, num_heads=2, num_rbf=2),
    non_hessian_subset_builder.dataset_info,
)

# set `hessian=True` so that the `MLIPNetwok` is wrapped inside a `HessianPredictor`
required_properties = Properties(hessian=True)

force_field = ForceField.from_mlip_network(
    mlip_network, required_properties=required_properties, seed=42
)

Setting the Hessian schedule to a constant weight `1000` (default is `0`) to enable Hessian-supervision. Using default schedules for energies and forces.

In [ ]:
optimizer = get_default_mlip_optimizer()
loss = MSELoss(hessian_weight_schedule=lambda _: 1000, extended_metrics=True)

In [ ]:
training_config = TrainingLoopConfig(num_epochs=10)

training_loop = TrainingLoop(
    train_dataset=combined_splits["train"],
    validation_dataset=combined_splits["valid"],
    force_field=force_field,
    loss=loss,
    optimizer=optimizer,
    config=training_config,
)

In [ ]:
training_loop.run()

## 4. Full Hessian Matrix prediction with the Hessian-trained MLIP

Using *mlip* to predict Hessians is as simple as performing a forward pass on a given graph.

In [ ]:
# On the same graph used previously, we clear the Hessian
# field again as it contains the non-Hessian model prediction
example_graph = example_graph.replace_nodes(hessian=None)

jitted_force_field_fun = jax.jit(training_loop.best_model)
prediction = jitted_force_field_fun(example_graph)

We can easily check that in the case of the model trained on Hessians, the predictor already includes Hessians as a required property which therefore (and unlike the previous model) does not need to be added.

In [ ]:
predicted_properties = dataclasses.fields(prediction)
predicted_property_names = [
    prop.name
    for prop in predicted_properties
    if getattr(prediction, prop.name) is not None
]
print(f"Predicted properties: {predicted_property_names}")
print(f"Predicted Hessian matrix shape: {prediction.hessian.shape}")

**We now compare the output of the models trained with and without Hessians:**

In [ ]:
mae_hessian = np.mean(np.abs(ref_hessian - prediction.hessian))
mae_no_hessian = np.mean(np.abs(ref_hessian - prediction_no_hessians.hessian))

print(f"MAE of Hessian model to reference Hessian: {mae_hessian}")
print(f"MAE of no-Hessian model to reference Hessian: {mae_no_hessian}")

For illustration we draw below the curvature distribution and the spectrum of the reference Hessians, when compared to the models trained with, and without Hessians.

In [ ]:
n_atoms = ref_hessian.shape[0]  # 21
matrix_size = n_atoms * 3  # 63

ref_2d = ref_hessian.reshape(matrix_size, matrix_size)
with_h_2d = prediction.hessian.reshape(matrix_size, matrix_size)
no_h_2d = prediction_no_hessians.hessian.reshape(matrix_size, matrix_size)


def get_sorted_eigenvalues(matrix_2d):
    # eigh expects a square symmetric 2D matrix
    eigenvalues = np.linalg.eigvalsh(matrix_2d)
    return np.sort(eigenvalues)


evals_ref = get_sorted_eigenvalues(ref_2d)
evals_with_h = get_sorted_eigenvalues(with_h_2d)
evals_no_h = get_sorted_eigenvalues(no_h_2d)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Smooth curvature profile spectrum
min_val = min(np.min(evals_ref), np.min(evals_with_h), np.min(evals_no_h))
max_val = max(np.max(evals_ref), np.max(evals_with_h), np.max(evals_no_h))
x_axis = np.linspace(min_val - 2, max_val + 2, 500)


def smooth_spectrum(eigenvalues, x_range, sigma=0.3):
    spectrum = np.zeros_like(x_range)
    for val in eigenvalues:
        spectrum += np.exp(-0.5 * ((x_range - val) / sigma) ** 2)
    return spectrum


spec_ref = smooth_spectrum(evals_ref, x_axis)
spec_with_h = smooth_spectrum(evals_with_h, x_axis)
spec_no_h = smooth_spectrum(evals_no_h, x_axis)

y_max = max(np.max(spec_ref), np.max(spec_with_h), np.max(spec_no_h)) * 1.05

axes[1].plot(x_axis, spec_ref, label="Reference Hessian", color="black", lw=2.5)
axes[0].plot(x_axis, spec_ref, label="Reference Hessian", color="black", lw=2.5)
axes[1].plot(
    x_axis,
    spec_with_h,
    label="Model (Trained with Hessian)",
    color="green",
    ls="--",
    lw=2,
)
axes[0].plot(
    x_axis,
    spec_no_h,
    label="Model (Trained without Hessian)",
    color="red",
    ls=":",
    lw=2,
)

for ax in axes:
    ax.set_xlim(min_val - 2, max_val + 2)
    ax.set_ylim(0, y_max)

axes[1].set_title("Hessian Eigenvalue Profile (Curvature Distribution)", fontsize=11)
axes[1].set_xlabel("Eigenvalue", fontsize=10)
axes[1].set_ylabel("Density of States", fontsize=10)
axes[1].legend(frameon=True)
axes[1].grid(alpha=0.3)

axes[0].set_title("Hessian Eigenvalue Profile (Curvature Distribution)", fontsize=11)
axes[0].set_xlabel("Eigenvalue", fontsize=10)

axes[0].set_ylabel("Density of States", fontsize=10)
axes[0].legend(frameon=True)
axes[0].grid(alpha=0.3)

plt.tight_layout()
plt.show()

additionally, we display the Hessian Eigenvalues (sorted by index) to evaluate the agreement of each model's actual Eigenvalues with the reference.

In [ ]:
mode_indices = np.arange(1, matrix_size + 1)

plt.figure(figsize=(7, 4.5))

plt.plot(mode_indices, evals_ref, label="Reference", color="black", lw=2.5, zorder=2)

plt.plot(
    mode_indices,
    evals_with_h,
    label="Trained with Hessian",
    color="green",
    ls="--",
    marker="o",
    markersize=4,
    alpha=0.8,
)

plt.plot(
    mode_indices,
    evals_no_h,
    label="Trained without Hessian",
    color="red",
    ls=":",
    marker="x",
    markersize=4,
    alpha=0.8,
)

plt.title("Hessian Eigenvalue Spectrum (Sorted Curvatures)", fontsize=11)
plt.xlabel("Eigenvalue Index", fontsize=10)
plt.ylabel("Eigenvalue", fontsize=10)
plt.legend(frameon=True, loc="upper left")
plt.grid(True, alpha=0.3, ls="--")

plt.tight_layout()
plt.show()